In [ ]:
# 0. Install / import dependencies
# torch, torchvision, pillow, requests, tqdm, pandas, numpy, matplotlib, scikit-learn
# are all preinstalled on Kaggle notebooks - no pip install needed.

import os
import io
import json
import time
import random
import glob
import concurrent.futures as cf
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms, models
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

ImageFile.LOAD_TRUNCATED_IMAGES = True  # don't crash on partially-downloaded images

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type != "cuda":
    print("WARNING: no GPU detected. Go to Settings -> Accelerator -> GPU T4 x2, then Save & re-run.")


In [ ]:
# 1. Paths - Kaggle input data is read-only, mounted under /kaggle/input/<dataset-name>/
# First, let's just see what's actually there so we can find the right folder.
print("Contents of /kaggle/input/:")
for p in sorted(glob.glob("/kaggle/input/*")):
    print(" -", p)


In [ ]:
# Try to auto-detect the dataset folder (matches 'furniture' or the json filenames inside it).
candidates = glob.glob("/kaggle/input/*furniture*") + glob.glob("/kaggle/input/*material*")

if not candidates:
    # fall back: scan every folder under /kaggle/input for train.json
    for p in glob.glob("/kaggle/input/*"):
        if os.path.isfile(os.path.join(p, "train.json")):
            candidates.append(p)
        else:
            # maybe nested one level deeper
            nested = glob.glob(os.path.join(p, "*", "train.json"))
            candidates.extend(os.path.dirname(n) for n in nested)

if candidates:
    DATA_DIR = Path(candidates[0])
    print("Auto-detected data dir:", DATA_DIR)
else:
    # MANUAL OVERRIDE: if auto-detect fails, look at the printed list from the
    # previous cell, copy the exact path you see there, and paste it below.
    DATA_DIR = Path("/kaggle/input/PASTE-EXACT-FOLDER-NAME-HERE")
    print("Auto-detect failed - set DATA_DIR manually above to one of the paths printed in the previous cell.")

print(list(DATA_DIR.glob("*")))

# /kaggle/working/ is the only writable, persisted-with-your-notebook location
IMG_DIR = Path("/kaggle/working/images")
for split in ["train", "validation", "test"]:
    (IMG_DIR / split).mkdir(parents=True, exist_ok=True)

TRAIN_JSON = DATA_DIR / "train.json"
VAL_JSON   = DATA_DIR / "validation.json"
TEST_JSON  = DATA_DIR / "test.json"
SAMPLE_SUB = DATA_DIR / "sample_submission_randomlabel.csv"

with open(TRAIN_JSON) as f:
    train_raw = json.load(f)
with open(VAL_JSON) as f:
    val_raw = json.load(f)
with open(TEST_JSON) as f:
    test_raw = json.load(f)

print("train images:", len(train_raw["images"]), "| train annotations:", len(train_raw["annotations"]))
print("val images:", len(val_raw["images"]), "| val annotations:", len(val_raw["annotations"]))
print("test images:", len(test_raw["images"]))

NUM_CLASSES = max(a["label_id"] for a in train_raw["annotations"])
print("num classes:", NUM_CLASSES)


In [ ]:
labels = [a["label_id"] for a in train_raw["annotations"]]
counts = pd.Series(labels).value_counts().sort_index()

plt.figure(figsize=(14,4))
plt.bar(counts.index, counts.values)
plt.xlabel("label_id"); plt.ylabel("num training images")
plt.title("Training class distribution (128 furniture categories)")
plt.show()

print("min/max/median images per class:", counts.min(), counts.max(), counts.median())


In [ ]:
def build_url_map(raw):
    # image_id -> first url in the list
    return {img["image_id"]: img["url"][0] for img in raw["images"]}

train_urls = build_url_map(train_raw)
val_urls   = build_url_map(val_raw)
test_urls  = build_url_map(test_raw)

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; furniture-classifier/1.0)"}

def download_one(args):
    image_id, url, out_dir = args
    out_path = out_dir / f"{image_id}.jpg"
    if out_path.exists() and out_path.stat().st_size > 0:
        return image_id, True
    for attempt in range(3):
        try:
            r = requests.get(url, headers=HEADERS, timeout=8)
            if r.status_code == 200 and len(r.content) > 500:
                img = Image.open(io.BytesIO(r.content)).convert("RGB")
                img.save(out_path, "JPEG", quality=90)
                return image_id, True
        except Exception:
            time.sleep(0.5)
    return image_id, False

def download_split(url_map, split_name, max_workers=32):
    out_dir = IMG_DIR / split_name
    tasks = [(iid, url, out_dir) for iid, url in url_map.items()]
    results = {}
    with cf.ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(download_one, t) for t in tasks]
        for fut in tqdm(cf.as_completed(futures), total=len(futures), desc=f"downloading {split_name}"):
            image_id, ok = fut.result()
            results[image_id] = ok
    n_ok = sum(results.values())
    print(f"{split_name}: {n_ok}/{len(results)} images downloaded successfully")
    return results

# NOTE: this can take a long time for the full 194k/6.4k/12.8k images.
# Start with a smaller max_workers if your connection is unstable, or
# subsample train_urls first while prototyping (see next cell).


In [ ]:
# OPTIONAL while prototyping: subsample the training set so the pipeline runs fast end-to-end.
# Comment this out (or set SUBSAMPLE=None) to use the full dataset for a real training run.
SUBSAMPLE = 20000  # e.g. 20k train images instead of 194k -- set to None for full run

if SUBSAMPLE is not None and SUBSAMPLE < len(train_urls):
    keep_ids = set(random.sample(list(train_urls.keys()), SUBSAMPLE))
    train_urls_use = {k: v for k, v in train_urls.items() if k in keep_ids}
else:
    train_urls_use = train_urls

print("training on", len(train_urls_use), "candidate images")


In [ ]:
train_status = download_split(train_urls_use, "train")
val_status   = download_split(val_urls, "validation")
test_status  = download_split(test_urls, "test")


In [ ]:
id2label_train = {a["image_id"]: a["label_id"] for a in train_raw["annotations"]}
id2label_val   = {a["image_id"]: a["label_id"] for a in val_raw["annotations"]}

train_items = [(iid, id2label_train[iid]) for iid, ok in train_status.items()
               if ok and iid in id2label_train]
val_items   = [(iid, id2label_val[iid]) for iid, ok in val_status.items()
               if ok and iid in id2label_val]
test_ids_ok    = [iid for iid, ok in test_status.items() if ok]
test_ids_fail  = [iid for iid, ok in test_status.items() if not ok]

print("usable train images:", len(train_items))
print("usable val images:", len(val_items))
print("usable test images:", len(test_ids_ok), "| missing test images:", len(test_ids_fail))

# most frequent class in train -> fallback prediction for images we could never download
FALLBACK_LABEL = pd.Series([l for _, l in train_items]).value_counts().idxmax()
print("fallback label for undownloadable test images:", FALLBACK_LABEL)


In [ ]:
IMG_SIZE = 224

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tfms = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class FurnitureDataset(Dataset):
    def __init__(self, items, img_dir, transform, has_labels=True):
        self.items = items  # list of (image_id, label) or list of image_id
        self.img_dir = img_dir
        self.transform = transform
        self.has_labels = has_labels

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        if self.has_labels:
            image_id, label = self.items[idx]
        else:
            image_id = self.items[idx]
            label = -1
        path = self.img_dir / f"{image_id}.jpg"
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (0, 0, 0))
        img = self.transform(img)
        # labels in the data are 1..128 -> convert to 0..127 for PyTorch
        return img, (label - 1 if self.has_labels else image_id)

train_ds = FurnitureDataset(train_items, IMG_DIR / "train", train_tfms, has_labels=True)
val_ds   = FurnitureDataset(val_items,   IMG_DIR / "validation", eval_tfms, has_labels=True)
test_ds  = FurnitureDataset(test_ids_ok, IMG_DIR / "test", eval_tfms, has_labels=False)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(len(train_ds), len(val_ds), len(test_ds))


In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

# freeze the backbone first, train only the new head for a couple epochs (fast warm-up)
for p in model.parameters():
    p.requires_grad = False

model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.fc.in_features, NUM_CLASSES),
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-4)


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    torch.set_grad_enabled(is_train)
    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        if is_train:
            optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        if is_train:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        all_preds.append(outputs.argmax(1).detach().cpu().numpy())
        all_labels.append(labels.detach().cpu().numpy())
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader.dataset), acc

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

EPOCHS_HEAD = 3
for epoch in range(EPOCHS_HEAD):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
    va_loss, va_acc = run_epoch(model, val_loader, criterion, optimizer=None)
    history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss);   history["val_acc"].append(va_acc)
    print(f"[head warm-up {epoch+1}/{EPOCHS_HEAD}] train_loss={tr_loss:.3f} train_acc={tr_acc:.3f} "
          f"val_loss={va_loss:.3f} val_acc={va_acc:.3f}")


In [ ]:
# Phase 2: unfreeze everything, fine-tune end-to-end at a lower LR
for p in model.parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=8)

EPOCHS_FINE = 8
best_val_acc = 0.0
CKPT_PATH = "/kaggle/working/best_model.pt"

for epoch in range(EPOCHS_FINE):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
    va_loss, va_acc = run_epoch(model, val_loader, criterion, optimizer=None)
    scheduler.step()
    history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss);   history["val_acc"].append(va_acc)
    print(f"[fine-tune {epoch+1}/{EPOCHS_FINE}] train_loss={tr_loss:.3f} train_acc={tr_acc:.3f} "
          f"val_loss={va_loss:.3f} val_acc={va_acc:.3f}")
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), CKPT_PATH)
        print(f"  -> saved new best checkpoint (val_acc={va_acc:.3f})")

print("Best validation accuracy:", best_val_acc)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy"); axes[1].legend()
plt.show()


In [ ]:
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()

pred_map = {}
with torch.no_grad():
    for imgs, image_ids in tqdm(test_loader, desc="predicting"):
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(1).cpu().numpy() + 1  # back to 1..128
        for iid, p in zip(image_ids.numpy(), preds):
            pred_map[int(iid)] = int(p)

# fill in the ones we couldn't download at all
for iid in test_ids_fail:
    pred_map[iid] = int(FALLBACK_LABEL)

sample = pd.read_csv(SAMPLE_SUB)
sample["predicted"] = sample["id"].map(pred_map)
assert sample["predicted"].isna().sum() == 0, "some ids missing predictions!"

sample.to_csv("/kaggle/working/submission.csv", index=False)
sample.head()


In [ ]:
# quick sanity check that the file Kaggle will pick up for scoring/submission looks right
check = pd.read_csv("/kaggle/working/submission.csv")
print(check.shape)
check.head()
